<a href="https://colab.research.google.com/github/sreelekha2196/Pixel-Seal-Hybrid-Watermarking-Framework-/blob/main/C2PA_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install c2pa-python cryptography

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 37.2 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/contentauth/c2pa-python /content/c2pa-python

Cloning into '/content/c2pa-python'...
remote: Enumerating objects: 2731, done.
remote: Counting objects: 100% (1020/1020), done.
remote: Compressing objects: 100% (327/327), done.
remote: Total 2731 (delta 889), reused 692 (delta 691), pack-reused 1711 (from 1)
Receiving objects: 100% (2731/2731), 21.50 MiB | 36.51 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [3]:
import os
import c2pa
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.backends import default_backend

fixtures_dir = "/content/c2pa-python/tests/fixtures/"
output_dir = "/content/c2pa_output/"
os.makedirs(output_dir, exist_ok=True)

print("c2pa version:", c2pa.sdk_version())

with open(fixtures_dir + "es256_certs.pem", "rb") as f:
    certs = f.read()
with open(fixtures_dir + "es256_private.key", "rb") as f:
    key = f.read()

def callback_signer_es256(data: bytes) -> bytes:
    private_key = serialization.load_pem_private_key(key, password=None, backend=default_backend())
    return private_key.sign(data, ec.ECDSA(hashes.SHA256()))

manifest_definition = {
    "claim_generator_info": [{"name": "pixelseal_c2pa_hybrid", "version": "0.0.1"}],
    "format": "image/jpeg",
    "title": "Test Signed Image",
    "ingredients": [],
    "assertions": [{
        "label": "c2pa.actions",
        "data": {"actions": [{"action": "c2pa.created", "digitalSourceType": "http://cv.iptc.org/newscodes/digitalsourcetype/digitalCreation"}]}
    }]
}

with c2pa.Context() as context:
    print("\nSigning the image file...")
    with c2pa.Signer.from_callback(callback_signer_es256, c2pa.C2paSigningAlg.ES256, certs.decode('utf-8'), "http://timestamp.digicert.com") as signer:
        with c2pa.Builder(manifest_definition, context) as builder:
            builder.sign_file(fixtures_dir + "A.jpg", output_dir + "A_signed.jpg", signer)

    print("\nReading signed image metadata:")
    with open(output_dir + "A_signed.jpg", "rb") as file:
        with c2pa.Reader("image/jpeg", file, context=context) as reader:
            print(reader.json())

print("\nExample completed successfully!")

c2pa version: 0.91.0

Signing the image file...

Reading signed image metadata:
{
  "active_manifest": "urn:c2pa:fdfa98a1-7a99-4d8c-b6e2-999005ccc7b0",
  "manifests": {
    "urn:c2pa:fdfa98a1-7a99-4d8c-b6e2-999005ccc7b0": {
      "claim_generator_info": [
        {
          "name": "pixelseal_c2pa_hybrid",
          "version": "0.0.1",
          "org.contentauth.c2pa_rs": "0.91.0"
        }
      ],
      "title": "Test Signed Image",
      "instance_id": "xmp.iid:813ee422-9736-4cdc-9be6-4e35ed8e41cb",
      "thumbnail": {
        "format": "image/jpeg",
        "identifier": "self#jumbf=/c2pa/urn:c2pa:fdfa98a1-7a99-4d8c-b6e2-999005ccc7b0/c2pa.assertions/c2pa.thumbnail.claim"
      },
      "assertions": [
        {
          "label": "c2pa.actions.v2",
          "data": {
            "actions": [
              {
                "action": "c2pa.created",
                "digitalSourceType": "http://cv.iptc.org/newscodes/digitalsourcetype/digitalCreation"
              }
            ]
